
# Trump Truth Social + UCDP Iran Event Timeline

This Colab replaces the R Markdown pipeline.

It does everything in one place:

1. Loads `trump_iran_event_windows_unique_dates.csv`.
2. Loads the UCDP Excel workbook.
3. Recalculates **Standard / Notable / High / Major / Context** event-day significance directly from the workbook.
4. Runs a Hugging Face **multi-label zero-shot classifier** on `content_text` only.
5. Scores four rhetorical frames:
   - **Threat**
   - **Victory**
   - **Diplomacy**
   - **Ceasefire / Ending**
6. Calculates event-balanced before / event-day / after averages.
7. Builds the connected interactive HTML:
   - UCDP event card above the event circle
   - event-day posts above the rail
   - day-before posts branching below-left
   - day-after posts branching below-right
   - wide desktop-style Truth Social cards
   - clickable event-date timeline across the bottom
   - Previous / Next event controls
   - event significance shown by marker size/color
8. Saves the scored data and HTML to Google Drive.

### Important

The classifier receives **only `content_text`**. It never sees the UCDP headline, fatalities, location, significance category, attachment description, or event metadata.

For speed, use a **T4 GPU** in Colab:

**Runtime → Change runtime type → T4 GPU**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Install packages

In [ ]:

!pip -q install transformers accelerate sentencepiece sentence-transformers openpyxl pandas numpy scikit-learn


## 2. Mount Google Drive

In [ ]:

from pathlib import Path
from google.colab import drive, files
import shutil
import os

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Trump_Iran_Project")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/Trump_Iran_Project



## 3. Find or upload the two source files

Required:

- `trump_iran_event_windows_unique_dates.csv`
- your UCDP workbook, e.g. `UCDP_Iran_US_or_Joint_US_Involvement_2026_through_July.xlsx`

The notebook first searches `MyDrive/Trump_Iran_Project`. If a file is missing, it asks you to upload it.


In [ ]:

def find_first(patterns):
    for pattern in patterns:
        hits = sorted(PROJECT_DIR.glob(pattern))
        if hits:
            return hits[-1]
    return None

def upload_one(destination_name, prompt):
    print(prompt)
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No file uploaded.")
    uploaded_name = next(iter(uploaded))
    destination = PROJECT_DIR / destination_name
    shutil.copy2(uploaded_name, destination)
    return destination

EVENT_CSV = find_first([
    "trump_iran_event_windows_unique_dates.csv",
    "*event_windows_unique_dates*.csv",
    "*unique_dates*.csv",
])

if EVENT_CSV is None:
    EVENT_CSV = upload_one(
        "trump_iran_event_windows_unique_dates.csv",
        "Upload trump_iran_event_windows_unique_dates.csv"
    )

UCDP_XLSX = find_first([
    "UCDP_Iran_US_or_Joint_US_Involvement_2026_through_July*.xlsx",
    "*UCDP*Iran*US*2026*.xlsx",
    "*UCDP*Iran*.xlsx",
])

if UCDP_XLSX is None:
    UCDP_XLSX = upload_one(
        "UCDP_Iran_US_or_Joint_US_Involvement_2026_through_July.xlsx",
        "Upload the UCDP Iran event workbook"
    )

print("Event-window CSV:", EVENT_CSV)
print("UCDP workbook:", UCDP_XLSX)


Event-window CSV: /content/drive/MyDrive/Trump_Iran_Project/trump_iran_event_windows_unique_dates.csv
UCDP workbook: /content/drive/MyDrive/Trump_Iran_Project/UCDP_Iran_US_or_Joint_US_Involvement_2026_through_July.xlsx


## 4. Load and validate the event-window data

In [ ]:

import pandas as pd
import numpy as np

events = pd.read_csv(EVENT_CSV)

required = {
    "truth_id",
    "content_text",
    "truth_url",
    "created_at_et",
    "event_date",
    "relative_day",
    "window_label",
}

missing = required - set(events.columns)
if missing:
    raise KeyError(f"Event-window CSV is missing required columns: {sorted(missing)}")

events["truth_id"] = events["truth_id"].astype(str)
events["content_text"] = events["content_text"].fillna("").astype(str)
events["event_date"] = pd.to_datetime(events["event_date"], errors="coerce").dt.normalize()
events["relative_day"] = pd.to_numeric(events["relative_day"], errors="coerce")

WINDOW_NAMES = {
    -1: "Day before",
     0: "Event day",
     1: "Day after",
}
events["window"] = events["relative_day"].map(WINDOW_NAMES).fillna(events["window_label"])

print("Event-window rows:", len(events))
print("Unique Truth Social posts:", events["truth_id"].nunique())
print("UCDP event dates:", events["event_date"].nunique())
display(events[["event_date", "window", "created_at_et", "content_text"]].head())


Event-window rows: 129
Unique Truth Social posts: 87
UCDP event dates: 26


,event_date,window,created_at_et,content_text
0,2026-02-28,Day after,2026-03-01 09:55:00-05:00,Trump’s bold move to rid the world of Iran’s e...
1,2026-02-28,Day after,2026-03-01 00:22:00-05:00,Iran just stated that they are going to hit ve...
2,2026-02-28,Event day,2026-02-28 16:37:00-05:00,"Khamenei, one of the most evil people in Histo..."
3,2026-02-28,Event day,2026-02-28 04:35:00-05:00,"Iran tried to interfere in 2020, 2024 election..."
4,2026-02-28,Day after,2026-03-01 00:25:00-05:00,Iran just stated that they are going to hit ve...



## 5. Recalculate UCDP event-day significance from the workbook

This avoids the misleading problem of assigning a multi-week cumulative death total to the first day of a long UCDP record.

For each event date in the Truth Social event-window CSV:

- **same-day record** = `date_start == event date == date_end`
- **spanning record** = starts on the event date but ends later

The classification rule is:

- **Major**: same-day `best` deaths ≥ 25 **or** same-day civilian deaths ≥ 10
- **High**: same-day `best` deaths ≥ 10 **or** at least 3 same-day UCDP records
- **Notable**: same-day `best` deaths ≥ 5 **or** at least 2 same-day UCDP records
- **Standard**: at least 1 same-day UCDP record
- **Context**: no same-day fatality record, but a multi-day UCDP record starts that date
- **Unclassified**: no matching workbook record

This is a transparent visualization rule, not an official UCDP severity scale.


In [ ]:

ucdp = pd.read_excel(UCDP_XLSX, sheet_name="US-involved events")

required_ucdp = {
    "date_start", "date_end", "best", "deaths_civilians",
    "us_involvement", "source_headline", "where_coordinates"
}
missing_ucdp = required_ucdp - set(ucdp.columns)
if missing_ucdp:
    raise KeyError(f"UCDP workbook is missing: {sorted(missing_ucdp)}")

ucdp["date_start"] = pd.to_datetime(ucdp["date_start"], errors="coerce").dt.normalize()
ucdp["date_end"] = pd.to_datetime(ucdp["date_end"], errors="coerce").dt.normalize()
ucdp["best"] = pd.to_numeric(ucdp["best"], errors="coerce").fillna(0)
ucdp["deaths_civilians"] = pd.to_numeric(
    ucdp["deaths_civilians"], errors="coerce"
).fillna(0)

event_dates = sorted(events["event_date"].dropna().unique())

def classify_significance(same_day_best, same_day_civilians, same_day_records, spanning_records):
    if same_day_best >= 25 or same_day_civilians >= 10:
        return "Major"
    if same_day_best >= 10 or same_day_records >= 3:
        return "High"
    if same_day_best >= 5 or same_day_records >= 2:
        return "Notable"
    if same_day_records >= 1:
        return "Standard"
    if spanning_records >= 1:
        return "Context"
    return "Unclassified"

sig_rows = []

for event_date in event_dates:
    same_day = ucdp[
        (ucdp["date_start"] == event_date) &
        (ucdp["date_end"] == event_date)
    ].copy()

    starts_today = ucdp[ucdp["date_start"] == event_date].copy()

    spanning = starts_today[
        starts_today["date_end"] > starts_today["date_start"]
    ].copy()

    same_day_records = len(same_day)
    same_day_best = float(same_day["best"].sum())
    same_day_civilians = float(same_day["deaths_civilians"].sum())
    spanning_records = len(spanning)

    level = classify_significance(
        same_day_best,
        same_day_civilians,
        same_day_records,
        spanning_records,
    )

    if same_day_records:
        reason_parts = [
            f"{same_day_records} same-day UCDP record" +
            ("" if same_day_records == 1 else "s")
        ]
        if same_day_best:
            reason_parts.append(f"best death estimate {same_day_best:g}")
        if same_day_civilians:
            reason_parts.append(f"{same_day_civilians:g} civilian deaths")
        reason = "; ".join(reason_parts)
    elif spanning_records:
        reason = (
            f"No same-day UCDP fatality record; {spanning_records} record"
            f"{'' if spanning_records == 1 else 's'} starting this date span later dates"
        )
    else:
        reason = "No matching UCDP record found for this event date"

    us_only = int((starts_today["us_involvement"] == "US").sum())
    joint = int((starts_today["us_involvement"] == "Joint US-Israel").sum())

    sig_rows.append({
        "event_date": event_date,
        "significance_level": level,
        "significance_reason": reason,
        "same_day_records": same_day_records,
        "same_day_best": same_day_best,
        "same_day_civilians": same_day_civilians,
        "spanning_records": spanning_records,
        "start_total_records": len(starts_today),
        "us_only_records": us_only,
        "joint_records": joint,
    })

significance = pd.DataFrame(sig_rows)

SIGNIFICANCE_CSV = PROJECT_DIR / "ucdp_event_day_significance.csv"
significance.to_csv(SIGNIFICANCE_CSV, index=False)

display(significance)
print("Saved:", SIGNIFICANCE_CSV)


,event_date,significance_level,significance_reason,same_day_records,same_day_best,same_day_civilians,spanning_records,start_total_records,us_only_records,joint_records
0,2026-02-28,Major,2 same-day UCDP records; best death estimate 1...,2,162.0,155.0,5,7,0,7
1,2026-03-04,Standard,1 same-day UCDP record; best death estimate 1,1,1.0,0.0,0,1,1,0
2,2026-03-08,Standard,1 same-day UCDP record; best death estimate 4,1,4.0,0.0,0,1,0,1
3,2026-03-20,Standard,1 same-day UCDP record; best death estimate 1,1,1.0,0.0,0,1,0,1
4,2026-03-27,Notable,2 same-day UCDP records; best death estimate 8,2,8.0,0.0,0,2,0,2
5,2026-03-29,Notable,1 same-day UCDP record; best death estimate 5,1,5.0,0.0,0,1,0,1
6,2026-03-30,High,1 same-day UCDP record; best death estimate 11...,1,11.0,5.0,0,1,0,1
7,2026-03-31,Standard,1 same-day UCDP record; best death estimate 4,1,4.0,0.0,0,1,0,1
8,2026-04-02,High,1 same-day UCDP record; best death estimate 13...,1,13.0,8.0,0,1,0,1
9,2026-04-04,High,3 same-day UCDP records; best death estimate 7...,3,7.0,1.0,0,3,0,3


Saved: /content/drive/MyDrive/Trump_Iran_Project/ucdp_event_day_significance.csv



## 6. Define the four zero-shot rhetorical frames

These are **multi-label**. One post can score high on more than one.

The longer descriptions are intentional; they tell the model what we mean by each frame more clearly than one-word labels would.


In [ ]:

FRAME_PROMPTS = {
    "Threat":
        "military threats, escalation, retaliation, ultimatums, warnings, strikes, or destruction directed at Iran",

    "Victory":
        "claims of victory, military success, Iranian weakness, defeat, surrender, decimation, or successful U.S. operations",

    "Diplomacy":
        "diplomacy, negotiations, talks, deals, conditions, agreements, or communication with Iran or allied governments",

    "Ceasefire / Ending":
        "a ceasefire, stopping military operations, ending the conflict, pausing attacks, or declaring the fighting essentially finished",
}

FRAME_NAMES = list(FRAME_PROMPTS)
CANDIDATE_LABELS = list(FRAME_PROMPTS.values())

FRAME_PROMPTS


{'Threat': 'military threats, escalation, retaliation, ultimatums, warnings, strikes, or destruction directed at Iran',
 'Victory': 'claims of victory, military success, Iranian weakness, defeat, surrender, decimation, or successful U.S. operations',
 'Diplomacy': 'diplomacy, negotiations, talks, deals, conditions, agreements, or communication with Iran or allied governments',
 'Ceasefire / Ending': 'a ceasefire, stopping military operations, ending the conflict, pausing attacks, or declaring the fighting essentially finished'}


## 7. Run or reuse the zero-shot scores

Model: `facebook/bart-large-mnli`

The notebook classifies each **unique `truth_id` once**, then merges the scores back onto every UCDP event window where that post appears.

The cache file is:

`trump_iran_zero_shot_scores.csv`

If that file already contains all current `truth_id` values and all four frames, the model does not run again.

**Interpretation:** these values are model entailment scores, not calibrated probabilities.


In [ ]:

import torch
from transformers import pipeline
from tqdm.auto import tqdm

SCORES_CSV = PROJECT_DIR / "trump_iran_zero_shot_scores.csv"

unique_posts = (
    events[["truth_id", "content_text"]]
    .drop_duplicates("truth_id")
    .reset_index(drop=True)
)

use_cache = False

if SCORES_CSV.exists():
    cached = pd.read_csv(SCORES_CSV)
    if "truth_id" in cached.columns:
        cached["truth_id"] = cached["truth_id"].astype(str)

        if (
            set(FRAME_NAMES).issubset(cached.columns)
            and set(unique_posts["truth_id"]).issubset(set(cached["truth_id"]))
        ):
            use_cache = True
            print("Using cached zero-shot scores:", SCORES_CSV)

if not use_cache:
    MODEL_NAME = "facebook/bart-large-mnli"
    device = 0 if torch.cuda.is_available() else -1

    print("Model:", MODEL_NAME)
    print("Device:", "GPU" if device == 0 else "CPU")

    classifier = pipeline(
        "zero-shot-classification",
        model=MODEL_NAME,
        device=device,
    )

    description_to_frame = {
        description: frame
        for frame, description in FRAME_PROMPTS.items()
    }

    output_rows = []
    batch_size = 8

    for start in tqdm(
        range(0, len(unique_posts), batch_size),
        desc="Zero-shot batches"
    ):
        batch = unique_posts["content_text"].iloc[
            start:start + batch_size
        ].tolist()

        results = classifier(
            batch,
            candidate_labels=CANDIDATE_LABELS,
            multi_label=True,
            hypothesis_template="This text expresses {}.",
            truncation=True,
        )

        if isinstance(results, dict):
            results = [results]

        for result in results:
            row = {frame: np.nan for frame in FRAME_NAMES}

            for description, score in zip(
                result["labels"],
                result["scores"]
            ):
                row[description_to_frame[description]] = float(score)

            output_rows.append(row)

    score_values = pd.DataFrame(output_rows)

    cached = pd.concat(
        [
            unique_posts.reset_index(drop=True),
            score_values.reset_index(drop=True),
        ],
        axis=1,
    )

    cached.to_csv(SCORES_CSV, index=False)
    print("Saved:", SCORES_CSV)

scores = cached.copy()

display(scores.head())


Model: facebook/bart-large-mnli
Device: CPU


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Zero-shot batches:   0%|          | 0/11 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/Trump_Iran_Project/trump_iran_zero_shot_scores.csv


,truth_id,content_text,Threat,Victory,Diplomacy,Ceasefire / Ending
0,116154493534592978,Trump’s bold move to rid the world of Iran’s e...,0.794440,0.935322,0.077754,0.892722
1,116152241303132753,Iran just stated that they are going to hit ve...,0.998625,0.969715,0.038067,0.024045
2,116150413051904167,"Khamenei, one of the most evil people in Histo...",0.660332,0.927226,0.008016,0.047326
3,116147572522796874,"Iran tried to interfere in 2020, 2024 election...",0.976250,0.668383,0.017946,0.321514
4,116152251973821428,Iran just stated that they are going to hit ve...,0.998591,0.979520,0.133881,0.030228


## 8. Merge scores and calculate event-balanced averages

In [ ]:

scored_events = events.merge(
    scores[["truth_id"] + FRAME_NAMES],
    on="truth_id",
    how="left",
)

# A post can appear in more than one event window when event dates are close.
# The event-date + truth_id pair is the analysis observation.
analysis_rows = (
    scored_events
    .drop_duplicates(["event_date", "truth_id"])
    .sort_values(["event_date", "relative_day", "created_at_et"])
    .reset_index(drop=True)
)

event_window_means = (
    analysis_rows
    .groupby(["event_date", "window"], observed=True)[FRAME_NAMES]
    .mean()
    .reset_index()
)

WINDOW_ORDER = ["Day before", "Event day", "Day after"]

overall_means = (
    event_window_means
    .groupby("window", observed=True)[FRAME_NAMES]
    .mean()
    .reindex(WINDOW_ORDER)
)

SCORED_EVENTS_CSV = PROJECT_DIR / "trump_iran_event_windows_zero_shot_scored.csv"
EVENT_MEANS_CSV = PROJECT_DIR / "trump_iran_event_balanced_frame_means.csv"

analysis_rows.to_csv(SCORED_EVENTS_CSV, index=False)
event_window_means.to_csv(EVENT_MEANS_CSV, index=False)

print("Event-balanced averages:")
display(overall_means.round(3))

print("Saved:", SCORED_EVENTS_CSV)
print("Saved:", EVENT_MEANS_CSV)


Event-balanced averages:


,Threat,Victory,Diplomacy,Ceasefire / Ending
window,,,,
Day before,0.548,0.673,0.223,0.343
Event day,0.524,0.761,0.209,0.328
Day after,0.485,0.700,0.272,0.391


Saved: /content/drive/MyDrive/Trump_Iran_Project/trump_iran_event_windows_zero_shot_scored.csv
Saved: /content/drive/MyDrive/Trump_Iran_Project/trump_iran_event_balanced_frame_means.csv



## 9. Build the connected interactive HTML

Design:

- one selected UCDP event at a time
- UCDP event card directly over the main event circle
- event-day posts above the rail
- day-before posts branching below-left
- day-after posts branching below-right
- wide social-post cards
- horizontal scrolling **inside each post group** when several posts exist
- clickable UCDP date timeline across the bottom
- marker size/color shows UCDP significance
- every post includes the four zero-shot scores


In [ ]:

import ast
import html
import re
from pathlib import Path

def esc(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return html.escape(str(value), quote=True)

def clean_preview(text, n=430):
    text = "" if text is None else str(text)
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return "Shared a link"
    return text if len(text) <= n else text[:n].rstrip() + "…"

def clean_short(text, n=480):
    text = "" if text is None else re.sub(r"\s+", " ", str(text)).strip()
    return text if len(text) <= n else text[:n].rstrip() + "…"

def first_media_url(raw):
    if raw is None:
        return ""
    try:
        if pd.isna(raw):
            return ""
    except Exception:
        pass

    text = str(raw).strip()
    if not text or text == "[]":
        return ""

    try:
        items = ast.literal_eval(text)
        if not isinstance(items, list) or not items:
            return ""

        item = items[0]
        kind = str(item.get("media_type", "") or "").lower()

        if kind == "video":
            return item.get("thumbnail_url") or ""

        url = item.get("archive_url") or item.get("thumbnail_url") or ""

        # Do not try to display mp4 directly as an image.
        if str(url).lower().split("?")[0].endswith(".mp4"):
            return item.get("thumbnail_url") or ""

        return url
    except Exception:
        return ""

FRAME_CSS = {
    "Threat": "threat",
    "Victory": "victory",
    "Diplomacy": "diplomacy",
    "Ceasefire / Ending": "ceasefire",
}

LEVEL_CSS = {
    "Standard": "standard",
    "Notable": "notable",
    "High": "high",
    "Major": "major",
    "Context": "context",
    "Unclassified": "unclassified",
}

def score_block(row):
    blocks = []

    for frame in FRAME_NAMES:
        value = row.get(frame, np.nan)
        value = 0.0 if pd.isna(value) else float(value)
        pct = max(0.0, min(100.0, 100.0 * value))

        label = "Ceasefire" if frame == "Ceasefire / Ending" else frame

        blocks.append(f"""
        <div class="score-row">
          <span>{esc(label)}</span>
          <span class="score-track">
            <span class="fill {FRAME_CSS[frame]}" style="width:{pct:.1f}%"></span>
          </span>
          <strong>{value:.2f}</strong>
        </div>
        """)

    return '<div class="score-area">' + "".join(blocks) + "</div>"

def truth_card(row):
    author = row.get("author_name", "Donald J. Trump")
    if not str(author).strip() or str(author).lower() == "nan":
        author = "Donald J. Trump"

    handle = row.get("author_handle", "realDonaldTrump")
    if not str(handle).strip() or str(handle).lower() == "nan":
        handle = "realDonaldTrump"

    text = row.get("content_text", "")
    post_url = row.get("truth_url", "")
    posted = row.get("created_at_et", "")
    window = row.get("window", "")
    media = first_media_url(row.get("attachments", ""))

    media_html = ""
    if media:
        media_html = (
            f'<img class="truth-media" src="{esc(media)}" '
            f'alt="Archived media attached to this Truth Social post">'
        )

    link_html = ""
    if isinstance(post_url, str) and post_url.strip():
        link_html = (
            f'<a class="truth-link" href="{esc(post_url)}" '
            f'target="_blank" rel="noopener noreferrer">'
            f'Open original Truth Social post ↗</a>'
        )

    return f"""
    <details class="truth-card">
      <summary>
        <div class="truth-head">
          <div class="avatar">DJT</div>
          <div class="identity">
            <div class="name">{esc(author)}</div>
            <div class="handle">@{esc(handle)}</div>
          </div>
        </div>

        {media_html}

        <div class="preview">{esc(clean_preview(text))}</div>

        {score_block(row)}

        <div class="meta">
          {esc(window)} · {esc(posted)}
        </div>
      </summary>

      <div class="full">
        <p>{esc(text)}</p>
        {link_html}
      </div>
    </details>
    """

def post_group(df):
    if df.empty:
        return '<div class="empty-posts">No matched posts</div>'

    df = df.sort_values("created_at_et")

    cards = "".join(
        truth_card(row)
        for _, row in df.iterrows()
    )

    return f"""
    <div class="post-scroll">
      <div class="post-track">
        {cards}
      </div>
    </div>
    """

sig_lookup = {
    pd.Timestamp(row["event_date"]).normalize(): row
    for _, row in significance.iterrows()
}

def get_sig(event_date):
    key = pd.Timestamp(event_date).normalize()
    if key in sig_lookup:
        return sig_lookup[key]

    return pd.Series({
        "significance_level": "Unclassified",
        "significance_reason": "No significance metadata available",
        "same_day_records": 0,
        "same_day_best": 0,
        "same_day_civilians": 0,
    })

def event_view(event_date, group, index):
    group = group.sort_values(["relative_day", "created_at_et"]).copy()

    before = group[group["relative_day"] == -1]
    dayof = group[group["relative_day"] == 0]
    after = group[group["relative_day"] == 1]

    first = group.iloc[0]
    sig = get_sig(event_date)

    level = str(sig["significance_level"])
    level_css = LEVEL_CSS.get(level, "unclassified")

    headline = clean_short(first.get("ucdp_source_headline", ""), 500)
    location = clean_short(first.get("ucdp_where_coordinates", ""), 250)

    event_date_ts = pd.Timestamp(event_date)
    before_date = (event_date_ts - pd.Timedelta(days=1)).strftime("%b %d")
    center_date = event_date_ts.strftime("%B %d, %Y")
    after_date = (event_date_ts + pd.Timedelta(days=1)).strftime("%b %d")

    return f"""
    <section class="event-view" data-event-index="{index}">
      <div class="event-stage">

        <div class="event-top">

          <div class="event-card">
            <div class="event-card-label">UCDP EVENT</div>
            <div class="event-card-title">{esc(headline)}</div>
            <div class="event-card-meta">{esc(location)}</div>

            <span class="sig-pill {level_css}">
              {esc(level)} event day
            </span>

            <div class="event-card-meta">
              {esc(sig["significance_reason"])}
            </div>
          </div>

          <div class="eventday-zone">
            <div class="eventday-label">
              Event day · {len(dayof)} {"post" if len(dayof) == 1 else "posts"}
            </div>
            {post_group(dayof)}
          </div>

        </div>

        <div class="rail-wrap">
          <div class="rail-line"></div>

          <div class="branch-node before-node"></div>
          <div class="center-node {level_css}"></div>
          <div class="branch-node after-node"></div>

          <div class="before-date">{esc(before_date)}</div>
          <div class="node-date">{esc(center_date)}</div>
          <div class="after-date">{esc(after_date)}</div>
        </div>

        <div class="branch-area">

          <div class="branch before-branch">
            <div class="branch-heading">
              <strong>Day before</strong>
              <span>{len(before)} {"post" if len(before) == 1 else "posts"}</span>
            </div>
            {post_group(before)}
          </div>

          <div class="branch after-branch">
            <div class="branch-heading">
              <strong>Day after</strong>
              <span>{len(after)} {"post" if len(after) == 1 else "posts"}</span>
            </div>
            {post_group(after)}
          </div>

        </div>

      </div>
    </section>
    """

sorted_dates = sorted(analysis_rows["event_date"].dropna().unique())

event_views_html = "".join(
    event_view(
        event_date,
        analysis_rows[analysis_rows["event_date"] == event_date],
        index,
    )
    for index, event_date in enumerate(sorted_dates)
)

def timeline_node(event_date, index):
    sig = get_sig(event_date)
    level = str(sig["significance_level"])
    level_css = LEVEL_CSS.get(level, "unclassified")
    reason = str(sig["significance_reason"])

    date_ts = pd.Timestamp(event_date)

    return f"""
    <button
      class="timeline-node {level_css}"
      type="button"
      data-event-index="{index}"
      title="{esc(date_ts.strftime('%B %d, %Y') + ' — ' + level + ': ' + reason)}"
    >
      <span class="timeline-dot"></span>
      <span class="timeline-date">{esc(date_ts.strftime('%b %d'))}</span>
    </button>
    """

timeline_nodes_html = "".join(
    timeline_node(event_date, index)
    for index, event_date in enumerate(sorted_dates)
)

select_options_html = "".join(
    f'<option value="{index}">{esc(pd.Timestamp(event_date).strftime("%B %d, %Y"))}</option>'
    for index, event_date in enumerate(sorted_dates)
)

def summary_table_html():
    rows = []

    for frame in FRAME_NAMES:
        label = "Ceasefire / Ending" if frame == "Ceasefire / Ending" else frame

        values = []
        for window in WINDOW_ORDER:
            value = overall_means.loc[window, frame] if window in overall_means.index else np.nan
            value = 0.0 if pd.isna(value) else float(value)
            pct = max(0, min(100, value * 100))

            values.append(f"""
            <td>
              <div class="summary-score">
                <span class="summary-track">
                  <span class="summary-fill {FRAME_CSS[frame]}" style="width:{pct:.1f}%"></span>
                </span>
                <strong>{value:.2f}</strong>
              </div>
            </td>
            """)

        rows.append(
            f"<tr><th>{esc(label)}</th>{''.join(values)}</tr>"
        )

    return f"""
    <table class="summary-table">
      <thead>
        <tr>
          <th>Frame</th>
          <th>Day before</th>
          <th>Event day</th>
          <th>Day after</th>
        </tr>
      </thead>
      <tbody>
        {''.join(rows)}
      </tbody>
    </table>
    """

SUMMARY_HTML = summary_table_html()

CSS = r"""
* { box-sizing: border-box; }

:root {
  --ink:#17202a;
  --muted:#667085;
  --border:#cfd7e1;
  --rail:#7f8b98;

  --standard:#8a96a3;
  --notable:#567a9c;
  --high:#9a6b43;
  --major:#9a4949;
  --context:#84759a;
  --unclassified:#9aa4af;

  --threat:#9a4949;
  --victory:#4e7657;
  --diplomacy:#4b719c;
  --ceasefire:#7d6594;
}

html,body {
  margin:0;
  padding:0;
  width:100%;
}

body {
  font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif;
  background:#f4f6f8;
  color:var(--ink);
  line-height:1.45;
}

.page {
  width:min(1400px, calc(100% - 24px));
  margin:0 auto;
  padding:24px 0 60px;
}

h1 { margin:0 0 5px; }
.subtitle { color:var(--muted); margin-bottom:18px; }

.panel {
  border:1px solid var(--border);
  border-radius:18px;
  background:white;
  margin:16px 0;
  overflow:hidden;
}

.panel-body { padding:18px; }

.note {
  border-left:4px solid #4d8ed6;
  background:#f2f7fd;
  padding:11px 14px;
  margin:14px 0;
}

.summary-scroll {
  overflow-x:auto;
  width:100%;
}

.summary-table {
  border-collapse:collapse;
  min-width:760px;
  width:100%;
}

.summary-table th,
.summary-table td {
  padding:10px 12px;
  border-bottom:1px solid #e6e9ed;
  text-align:left;
}

.summary-score {
  display:grid;
  grid-template-columns:1fr 38px;
  gap:8px;
  align-items:center;
}

.summary-track,
.score-track {
  display:block;
  height:8px;
  background:#e5e7eb;
  border-radius:999px;
  overflow:hidden;
}

.summary-fill,
.fill {
  display:block;
  height:100%;
  border-radius:999px;
}

.threat { background:var(--threat); }
.victory { background:var(--victory); }
.diplomacy { background:var(--diplomacy); }
.ceasefire { background:var(--ceasefire); }

.viewer-toolbar {
  display:flex;
  align-items:center;
  gap:9px;
  flex-wrap:wrap;
  padding:12px 14px;
  border-bottom:1px solid var(--border);
  background:#eef2f6;
}

.viewer-toolbar button,
.viewer-toolbar select {
  min-height:42px;
  border:1px solid #9da8b4;
  border-radius:9px;
  background:white;
  color:var(--ink);
  font-size:14px;
}

.viewer-toolbar button {
  padding:8px 13px;
  font-weight:800;
  cursor:pointer;
}

.viewer-toolbar button:disabled {
  opacity:.4;
  cursor:default;
}

.viewer-toolbar select {
  min-width:230px;
  padding:7px 10px;
}

.event-counter { font-weight:850; }

.toolbar-note {
  margin-left:auto;
  color:var(--muted);
  font-size:12px;
  font-weight:700;
}

.event-view { display:none; }
.event-view.is-active { display:block; }

.event-stage {
  position:relative;
  width:100%;
  padding:22px 24px 30px;
  background:#f8fafc;
}

.event-top {
  position:relative;
  width:100%;
  padding-bottom:50px;
  text-align:center;
}

.event-card {
  width:min(760px, calc(100% - 20px));
  margin:0 auto 16px;
  padding:16px 18px;
  border:2px solid #8d98a4;
  border-radius:17px;
  background:white;
  text-align:left;
}

.event-card-label {
  color:var(--muted);
  font-size:11px;
  font-weight:900;
  letter-spacing:.09em;
}

.event-card-title {
  margin-top:5px;
  font-size:17px;
  font-weight:850;
  line-height:1.38;
}

.event-card-meta {
  margin-top:7px;
  color:var(--muted);
  font-size:13px;
}

.sig-pill {
  display:inline-block;
  margin-top:9px;
  padding:5px 9px;
  border-radius:999px;
  color:white;
  font-size:11px;
  font-weight:850;
}

.sig-pill.standard { background:var(--standard); }
.sig-pill.notable { background:var(--notable); }
.sig-pill.high { background:var(--high); }
.sig-pill.major { background:var(--major); }
.sig-pill.context { background:var(--context); }
.sig-pill.unclassified { background:var(--unclassified); }

.eventday-zone {
  position:relative;
  width:100%;
  margin-top:6px;
}

.eventday-zone::after {
  content:"";
  position:absolute;
  left:50%;
  bottom:-50px;
  width:2px;
  height:48px;
  transform:translateX(-50%);
  background:var(--rail);
}

.eventday-label {
  margin-bottom:8px;
  color:var(--muted);
  font-size:12px;
  font-weight:850;
  text-transform:uppercase;
  letter-spacing:.06em;
}

.post-scroll {
  width:100%;
  overflow-x:auto;
  overflow-y:hidden;
  padding:3px 4px 12px;
  scrollbar-width:thin;
  scrollbar-color:#7d8793 #e5e9ee;
}

.post-scroll::-webkit-scrollbar { height:11px; }
.post-scroll::-webkit-scrollbar-track { background:#e5e9ee; border-radius:999px; }
.post-scroll::-webkit-scrollbar-thumb { background:#7d8793; border-radius:999px; }

.post-track {
  display:flex;
  gap:16px;
  width:max-content;
  min-width:100%;
}

.eventday-zone .post-track { justify-content:center; }

.rail-wrap {
  position:relative;
  height:92px;
}

.rail-line {
  position:absolute;
  left:6%;
  right:6%;
  top:38px;
  height:4px;
  border-radius:999px;
  background:var(--rail);
}

.center-node,
.branch-node {
  position:absolute;
  top:38px;
  transform:translate(-50%, -50%);
  border-radius:50%;
  background:white;
}

.center-node {
  left:50%;
  width:30px;
  height:30px;
  border:7px solid var(--standard);
}

.center-node.standard { border-color:var(--standard); }
.center-node.notable { border-color:var(--notable); }
.center-node.high { border-color:var(--high); }
.center-node.major {
  width:38px;
  height:38px;
  border-width:9px;
  border-color:var(--major);
}
.center-node.context { border-color:var(--context); }
.center-node.unclassified { border-color:var(--unclassified); }

.branch-node {
  width:17px;
  height:17px;
  border:4px solid #8a96a3;
}

.before-node { left:24%; }
.after-node { left:76%; }

.node-date {
  position:absolute;
  top:61px;
  left:50%;
  transform:translateX(-50%);
  white-space:nowrap;
  font-size:13px;
  font-weight:850;
}

.before-date,
.after-date {
  position:absolute;
  top:62px;
  color:var(--muted);
  font-size:12px;
  font-weight:750;
}

.before-date { left:24%; transform:translateX(-50%); }
.after-date { left:76%; transform:translateX(-50%); }

.branch-area {
  display:grid;
  grid-template-columns:minmax(0,1fr) minmax(0,1fr);
  gap:28px;
  width:100%;
}

.branch {
  position:relative;
  min-width:0;
  padding-top:34px;
}

.branch::before {
  content:"";
  position:absolute;
  top:-44px;
  width:2px;
  height:74px;
  background:var(--rail);
}

.before-branch::before { left:46%; }
.after-branch::before { left:54%; }

.branch-heading {
  display:flex;
  align-items:baseline;
  justify-content:center;
  gap:7px;
  margin-bottom:10px;
}

.branch-heading strong { font-size:14px; }
.branch-heading span { color:var(--muted); font-size:12px; }

.before-branch .post-track { justify-content:flex-end; }
.after-branch .post-track { justify-content:flex-start; }

.truth-card {
  flex:0 0 560px;
  width:560px;
  min-width:560px;
  max-width:560px;
  overflow:hidden;
  border:1px solid #ced6df;
  border-radius:18px;
  background:white;
  box-shadow:0 3px 12px rgba(15,23,42,.06);
}

.truth-card summary {
  display:block;
  width:100%;
  padding:16px 18px;
  list-style:none;
  cursor:pointer;
}

.truth-card summary::-webkit-details-marker { display:none; }

.truth-head {
  display:flex;
  align-items:center;
  gap:11px;
  margin-bottom:10px;
}

.avatar {
  display:flex;
  width:46px;
  height:46px;
  flex:0 0 46px;
  align-items:center;
  justify-content:center;
  border-radius:50%;
  background:#6688d0;
  color:white;
  font-size:12px;
  font-weight:900;
}

.name {
  font-size:16px;
  font-weight:850;
  line-height:1.15;
}

.handle {
  margin-top:2px;
  color:var(--muted);
  font-size:12px;
}

.truth-media {
  width:100%;
  max-height:330px;
  object-fit:cover;
  border-radius:14px;
  border:1px solid #e2e7ec;
  margin:4px 0 10px;
}

.preview {
  margin:9px 0 12px;
  font-size:15px;
  line-height:1.46;
  overflow-wrap:anywhere;
  display:-webkit-box;
  -webkit-box-orient:vertical;
  -webkit-line-clamp:4;
  overflow:hidden;
}

.score-area {
  display:grid;
  grid-template-columns:minmax(0,1fr) minmax(0,1fr);
  column-gap:18px;
  row-gap:7px;
  margin-top:10px;
  padding-top:10px;
  border-top:1px solid #eceff2;
}

.score-row {
  display:grid;
  grid-template-columns:75px minmax(0,1fr) 32px;
  gap:6px;
  align-items:center;
  font-size:10px;
}

.meta {
  margin-top:11px;
  color:var(--muted);
  font-size:11px;
}

.full {
  width:100%;
  padding:4px 18px 18px;
  border-top:1px solid #e7ebef;
  font-size:14px;
  line-height:1.55;
  overflow-wrap:anywhere;
}

.truth-link {
  display:inline-block;
  margin-top:6px;
  font-weight:800;
}

.empty-posts {
  width:100%;
  padding:24px;
  color:#98a2b3;
  text-align:center;
  font-size:13px;
}

.global-timeline {
  padding:20px 14px 16px;
  border-top:1px solid var(--border);
  background:white;
}

.global-timeline-title {
  display:flex;
  align-items:center;
  justify-content:space-between;
  gap:12px;
  margin-bottom:10px;
}

.timeline-legend {
  display:flex;
  gap:9px;
  flex-wrap:wrap;
  color:var(--muted);
  font-size:10px;
}

.legend-item {
  display:flex;
  align-items:center;
  gap:4px;
}

.legend-dot {
  width:10px;
  height:10px;
  border-radius:50%;
}

.timeline-scroll {
  width:100%;
  overflow-x:auto;
  overflow-y:hidden;
  padding:12px 10px 18px;
  scrollbar-width:thin;
}

.timeline-bar {
  position:relative;
  display:flex;
  align-items:flex-start;
  gap:18px;
  width:max-content;
  min-width:100%;
  padding-top:7px;
}

.timeline-bar::before {
  content:"";
  position:absolute;
  left:12px;
  right:12px;
  top:21px;
  height:3px;
  border-radius:999px;
  background:#b7c0ca;
}

.timeline-node {
  position:relative;
  display:flex;
  flex-direction:column;
  align-items:center;
  min-width:58px;
  padding:0;
  border:0;
  background:transparent;
  cursor:pointer;
  z-index:2;
}

.timeline-dot {
  display:block;
  width:16px;
  height:16px;
  border:4px solid white;
  border-radius:50%;
  box-shadow:0 0 0 2px var(--standard);
  background:var(--standard);
}

.timeline-node.notable .timeline-dot {
  width:19px;
  height:19px;
  background:var(--notable);
  box-shadow:0 0 0 2px var(--notable);
}

.timeline-node.high .timeline-dot {
  width:22px;
  height:22px;
  background:var(--high);
  box-shadow:0 0 0 2px var(--high);
}

.timeline-node.major .timeline-dot {
  width:27px;
  height:27px;
  background:var(--major);
  box-shadow:0 0 0 2px var(--major);
}

.timeline-node.context .timeline-dot {
  background:white;
  box-shadow:0 0 0 2px var(--context);
}

.timeline-node.unclassified .timeline-dot {
  background:white;
  box-shadow:0 0 0 2px var(--unclassified);
}

.timeline-date {
  margin-top:8px;
  color:var(--muted);
  font-size:10px;
  font-weight:750;
  white-space:nowrap;
}

.timeline-node.is-active .timeline-date {
  color:var(--ink);
  font-weight:900;
}

.timeline-node.is-active .timeline-dot {
  outline:3px solid rgba(32,40,51,.22);
  outline-offset:3px;
}

@media(max-width:900px) {
  .page { width:calc(100% - 12px); }

  .toolbar-note {
    width:100%;
    margin-left:0;
  }

  .viewer-toolbar select { width:100%; }

  .branch-area { grid-template-columns:1fr; }

  .before-branch::before,
  .after-branch::before { display:none; }

  .truth-card {
    flex-basis:86vw;
    width:86vw;
    min-width:86vw;
    max-width:86vw;
  }

  .score-area { grid-template-columns:1fr; }
}
"""

JS = r"""
(function () {
  const views = Array.from(document.querySelectorAll(".event-view"));
  const timelineNodes = Array.from(document.querySelectorAll(".timeline-node"));

  const prev = document.getElementById("prevEvent");
  const next = document.getElementById("nextEvent");
  const select = document.getElementById("eventSelect");
  const counter = document.getElementById("eventCounter");
  const timelineScroll = document.getElementById("timelineScroll");

  if (views.length === 0) return;

  let index = 0;

  function showEvent(newIndex) {
    index = Math.max(0, Math.min(views.length - 1, newIndex));

    views.forEach(function (view, i) {
      view.classList.toggle("is-active", i === index);
    });

    timelineNodes.forEach(function (node, i) {
      node.classList.toggle("is-active", i === index);
    });

    if (select) select.value = String(index);
    if (counter) counter.textContent = (index + 1) + " of " + views.length;
    if (prev) prev.disabled = index === 0;
    if (next) next.disabled = index === views.length - 1;

    views[index].querySelectorAll(".post-scroll").forEach(function (row) {
      row.scrollLeft = 0;
    });

    if (timelineScroll && timelineNodes[index]) {
      const node = timelineNodes[index];
      const desired =
        node.offsetLeft -
        timelineScroll.clientWidth / 2 +
        node.clientWidth / 2;

      timelineScroll.scrollTo({
        left: Math.max(0, desired),
        behavior: "smooth"
      });
    }
  }

  if (prev) {
    prev.addEventListener("click", function () {
      showEvent(index - 1);
    });
  }

  if (next) {
    next.addEventListener("click", function () {
      showEvent(index + 1);
    });
  }

  if (select) {
    select.addEventListener("change", function () {
      showEvent(Number(this.value));
    });
  }

  timelineNodes.forEach(function (node, nodeIndex) {
    node.addEventListener("click", function () {
      showEvent(nodeIndex);
    });
  });

  showEvent(0);
})();
"""

REPORT_HTML = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Trump Truth Social Rhetoric Around UCDP Iran Events</title>
<style>{CSS}</style>
</head>
<body>
<div class="page">

<h1>Trump Truth Social Rhetoric Around UCDP Iran Events</h1>
<div class="subtitle">
Connected event timeline with four-label zero-shot rhetorical analysis
</div>

<div class="panel">
  <div class="panel-body">
    <h2>Event-balanced rhetorical comparison</h2>
    <p>
      Posts are averaged within each UCDP event date/window first, then those
      event-level means are averaged. This keeps a very busy posting day from
      automatically dominating the overall result.
    </p>

    <div class="summary-scroll">
      {SUMMARY_HTML}
    </div>

    <div class="note">
      Zero-shot scores are model entailment scores, not calibrated probabilities.
      One post can score highly on more than one frame.
    </div>
  </div>
</div>

<div class="panel">

  <div class="viewer-toolbar">
    <button id="prevEvent" type="button">← Previous event</button>
    <button id="nextEvent" type="button">Next event →</button>

    <select id="eventSelect">
      {select_options_html}
    </select>

    <strong id="eventCounter" class="event-counter">
      1 of {len(sorted_dates)}
    </strong>

    <span class="toolbar-note">
      Circle size/color reflects UCDP event-day significance
    </span>
  </div>

  <div id="eventViews">
    {event_views_html}
  </div>

  <div class="global-timeline">

    <div class="global-timeline-title">
      <strong>UCDP event-date timeline</strong>

      <div class="timeline-legend">
        <span class="legend-item">
          <span class="legend-dot" style="background:var(--standard)"></span>
          Standard
        </span>
        <span class="legend-item">
          <span class="legend-dot" style="background:var(--notable)"></span>
          Notable
        </span>
        <span class="legend-item">
          <span class="legend-dot" style="background:var(--high)"></span>
          High
        </span>
        <span class="legend-item">
          <span class="legend-dot" style="background:var(--major)"></span>
          Major
        </span>
        <span class="legend-item">
          <span class="legend-dot" style="background:white;border:2px solid var(--context)"></span>
          Context
        </span>
      </div>
    </div>

    <div id="timelineScroll" class="timeline-scroll">
      <div class="timeline-bar">
        {timeline_nodes_html}
      </div>
    </div>

  </div>
</div>

<div class="panel">
  <div class="panel-body">
    <h2>How significance is calculated</h2>
    <p>
      This is a transparent visualization rule calculated from the supplied UCDP
      workbook. It is <strong>not</strong> an official UCDP severity category.
    </p>
    <ul>
      <li><strong>Major:</strong> same-day best death estimate ≥ 25 or civilian deaths ≥ 10.</li>
      <li><strong>High:</strong> same-day best estimate ≥ 10 or at least 3 same-day UCDP records.</li>
      <li><strong>Notable:</strong> same-day best estimate ≥ 5 or at least 2 same-day records.</li>
      <li><strong>Standard:</strong> another same-day UCDP event date.</li>
      <li><strong>Context:</strong> a multi-day UCDP record starts that day, but there is no same-day fatality record to assign to that date.</li>
    </ul>
  </div>
</div>

</div>

<script>{JS}</script>
</body>
</html>
"""

HTML_PATH = PROJECT_DIR / "Trump_Iran_Connected_ZeroShot_Timeline.html"
HTML_PATH.write_text(REPORT_HTML, encoding="utf-8")

print("Saved HTML:", HTML_PATH)
print("Size:", f"{HTML_PATH.stat().st_size / 1024:.1f} KB")


Saved HTML: /content/drive/MyDrive/Trump_Iran_Project/Trump_Iran_Connected_ZeroShot_Timeline.html
Size: 393.3 KB



## 10. Preview the HTML in Colab

This preview is useful for checking the result, but the downloaded HTML in a normal browser will usually feel better for the interactive timeline.


In [ ]:

from IPython.display import HTML, display

display(HTML(HTML_PATH.read_text(encoding="utf-8")))


Frame,Day before,Event day,Day after
Threat,0.55,0.52,0.48
Victory,0.67,0.76,0.70
Diplomacy,0.22,0.21,0.27
Ceasefire / Ending,0.34,0.33,0.39


## 11. Download the finished HTML

In [ ]:

from google.colab import files

files.download(str(HTML_PATH))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## 12. Optional extra text analysis

The timeline and zero-shot analysis above are the main pipeline.

If you want to recreate the other exploratory analyses we discussed, set:

```python
RUN_EXTRA_TEXT_ANALYSIS = True
```

The optional cell calculates:

- distinctive TF-IDF terms by event-window day
- keyness-style log2 word-frequency ratios
- sentence-embedding semantic similarity to each event day's language
- repeated-messaging similarity within each event window


In [ ]:

RUN_EXTRA_TEXT_ANALYSIS = False

if not RUN_EXTRA_TEXT_ANALYSIS:
    print("Extra TF-IDF / keyness / semantic analysis skipped.")
else:
    import re
    from collections import Counter
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    # ------------------------------------------------------------------
    # TF-IDF by window
    # ------------------------------------------------------------------
    window_documents = (
        analysis_rows
        .groupby("window", observed=True)["content_text"]
        .apply(lambda x: " ".join(x.astype(str)))
        .reindex(WINDOW_ORDER)
        .fillna("")
    )

    vectorizer = TfidfVectorizer(
        stop_words="english",
        lowercase=True,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z']+\b",
    )

    matrix = vectorizer.fit_transform(window_documents.values)
    terms = np.array(vectorizer.get_feature_names_out())

    tfidf_rows = []

    for i, window in enumerate(window_documents.index):
        values = matrix[i].toarray().ravel()
        top_idx = values.argsort()[::-1][:20]

        for rank, idx in enumerate(top_idx, start=1):
            if values[idx] <= 0:
                continue

            tfidf_rows.append({
                "window": window,
                "rank": rank,
                "word": terms[idx],
                "tfidf": float(values[idx]),
            })

    tfidf_results = pd.DataFrame(tfidf_rows)
    tfidf_results.to_csv(
        PROJECT_DIR / "trump_iran_tfidf_by_window.csv",
        index=False,
    )

    # ------------------------------------------------------------------
    # Keyness-style log2 ratio
    # ------------------------------------------------------------------
    def words(text):
        return re.findall(r"[A-Za-z][A-Za-z']+", str(text).lower())

    keyness_rows = []

    for target_window in WINDOW_ORDER:
        target_texts = analysis_rows.loc[
            analysis_rows["window"] == target_window,
            "content_text"
        ]

        other_texts = analysis_rows.loc[
            analysis_rows["window"] != target_window,
            "content_text"
        ]

        target_counts = Counter(
            token
            for text in target_texts
            for token in words(text)
        )

        other_counts = Counter(
            token
            for text in other_texts
            for token in words(text)
        )

        total_target = sum(target_counts.values())
        total_other = sum(other_counts.values())

        vocab = set(target_counts) | set(other_counts)

        for word in vocab:
            total_n = target_counts[word] + other_counts[word]

            if total_n < 2:
                continue

            target_rate = 1000 * (target_counts[word] + 0.5) / (total_target + 0.5)
            other_rate = 1000 * (other_counts[word] + 0.5) / (total_other + 0.5)

            keyness_rows.append({
                "target_window": target_window,
                "word": word,
                "target_n": target_counts[word],
                "other_n": other_counts[word],
                "log2_ratio": float(np.log2(target_rate / other_rate)),
            })

    keyness = pd.DataFrame(keyness_rows)

    top_keyness = (
        keyness
        .sort_values(
            ["target_window", "log2_ratio"],
            ascending=[True, False],
        )
        .groupby("target_window")
        .head(20)
    )

    top_keyness.to_csv(
        PROJECT_DIR / "trump_iran_keyness_by_window.csv",
        index=False,
    )

    # ------------------------------------------------------------------
    # Sentence-embedding semantic similarity
    # ------------------------------------------------------------------
    embed_model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

    unique_for_embedding = (
        analysis_rows[["truth_id", "content_text"]]
        .drop_duplicates("truth_id")
        .reset_index(drop=True)
    )

    vectors = embed_model.encode(
        unique_for_embedding["content_text"].tolist(),
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    vector_lookup = {
        str(unique_for_embedding.loc[i, "truth_id"]): vectors[i]
        for i in range(len(unique_for_embedding))
    }

    semantic = analysis_rows.copy()
    semantic["similarity_to_event_day"] = np.nan
    semantic["repetition_similarity"] = np.nan

    for event_date, idx in semantic.groupby("event_date").groups.items():
        idx = list(idx)
        group = semantic.loc[idx].copy()

        group_vectors = np.vstack([
            vector_lookup[str(tid)]
            for tid in group["truth_id"]
        ])

        event_positions = np.where(
            group["relative_day"].to_numpy() == 0
        )[0]

        if len(event_positions):
            centroid = group_vectors[event_positions].mean(axis=0, keepdims=True)
            centroid = centroid / np.linalg.norm(centroid)

            semantic.loc[
                idx,
                "similarity_to_event_day"
            ] = cosine_similarity(
                group_vectors,
                centroid
            ).ravel()

        if len(group_vectors) >= 2:
            pairwise = cosine_similarity(group_vectors)
            np.fill_diagonal(pairwise, np.nan)

            semantic.loc[
                idx,
                "repetition_similarity"
            ] = np.nanmax(pairwise, axis=1)

    semantic.to_csv(
        PROJECT_DIR / "trump_iran_semantic_similarity.csv",
        index=False,
    )

    print("Extra analysis saved to:", PROJECT_DIR)
    display(tfidf_results.head(20))
    display(top_keyness.head(20))
    display(
        semantic[
            [
                "event_date",
                "window",
                "content_text",
                "similarity_to_event_day",
                "repetition_similarity",
            ]
        ].head()
    )


Extra TF-IDF / keyness / semantic analysis skipped.



# Outputs

After running the notebook, `MyDrive/Trump_Iran_Project` should contain:

- `Trump_Iran_Connected_ZeroShot_Timeline.html`
- `trump_iran_zero_shot_scores.csv`
- `trump_iran_event_windows_zero_shot_scored.csv`
- `trump_iran_event_balanced_frame_means.csv`
- `ucdp_event_day_significance.csv`

If you enable the optional analysis:

- `trump_iran_tfidf_by_window.csv`
- `trump_iran_keyness_by_window.csv`
- `trump_iran_semantic_similarity.csv`

The zero-shot score CSV is deliberately cached. If you later change the four frame definitions and want fresh scores, delete `trump_iran_zero_shot_scores.csv` from the project folder and rerun the model section.
